In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import re


2026-01-07 07:21:24.163774: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767770484.177699      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767770484.182024      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 07:21:24.200383: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
def pythonic_loader_train():
    IMAGES_PATH = "/kaggle/input/petfinder-pawpularity-score/train/"
    CSV_PATH = "/kaggle/input/petfinder-pawpularity-score/train.csv"
    
    relevant_columns = ['Subject Focus','Eyes','Face','Near','Action','Accessory','Group','Collage','Human','Occlusion','Info','Blur']
    target = 'Pawpularity'
    
    image_names = os.listdir(IMAGES_PATH)
    image_names = image_names[:int(len(image_names)*0.95)]
    np.random.shuffle(image_names)
    metadata_csv = pd.read_csv(CSV_PATH)
    
    for i in image_names:
        img = tf.keras.utils.load_img(path = IMAGES_PATH+i,
                color_mode="rgb",
                target_size=(256,256))
        img = np.array(img)
        
        metadata = metadata_csv[metadata_csv['Id']==i[:-4]]
        features = metadata[relevant_columns].values[0]
        
        y = metadata[target].values[0]
        
        yield({"Image":img,"Feature":features}, y)


In [3]:
def pythonic_loader_test():
    IMAGES_PATH = "/kaggle/input/petfinder-pawpularity-score/train/"
    CSV_PATH = "/kaggle/input/petfinder-pawpularity-score/train.csv"
    
    relevant_columns = ['Subject Focus','Eyes','Face','Near','Action','Accessory','Group','Collage','Human','Occlusion','Info','Blur']
    target = 'Pawpularity'
    
    image_names = os.listdir(IMAGES_PATH)
    image_names = image_names[int(len(image_names)*0.95):]
    np.random.shuffle(image_names)
    metadata_csv = pd.read_csv(CSV_PATH)
    
    for i in image_names:
        img = tf.keras.utils.load_img(path = IMAGES_PATH+i,
                color_mode="rgb",
                target_size=(256,256))
        img = np.array(img)
        
        metadata = metadata_csv[metadata_csv['Id']==i[:-4]]
        features = metadata[relevant_columns].values[0]
        
        y = metadata[target].values[0]
        
        yield({"Image":img,"Feature":features}, y)


In [4]:
train_loader = tf.data.Dataset.from_generator(
            pythonic_loader_train,
            output_types=({'Image': tf.int64,
            'Feature': tf.int64},
               tf.int64))


Instructions for updating:
Use output_signature instead


2026-01-07 07:21:31.548923: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1767770491.549400      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


In [5]:
test_loader = tf.data.Dataset.from_generator(
                pythonic_loader_train,
                output_types=({'Image': tf.int64,
                'Feature': tf.int64},
               tf.int64))


In [6]:
input_image = tf.keras.Input(shape=(256,256,3))
input_meta = tf.keras.Input(shape=(12))

l2 = tf.keras.layers.MaxPool2D((2,2))(input_image)
l3 = tf.keras.layers.Conv2D(8,(3,3),activation='relu')(l2)
l4 = tf.keras.layers.MaxPool2D((2,2))(l3)
l5 = tf.keras.layers.Conv2D(16,(3,3),activation='relu')(l4)
l6 = tf.keras.layers.MaxPool2D((2,2))(l5)
l7 = tf.keras.layers.Conv2D(32,(3,3),activation='relu')(l6)
l_mid1 = tf.keras.layers.MaxPool2D((2,2))(l7)
l_mid2 = tf.keras.layers.Conv2D(64,(3,3),activation='relu')(l_mid1)
l8 = tf.keras.layers.Flatten()(l_mid2)
l10 = tf.keras.layers.Dense(1024, activation="gelu")(l8)

combined = tf.keras.layers.concatenate([l10, input_meta])
l12 = tf.keras.layers.Dense(512, activation="gelu")(combined)
bn1 = tf.keras.layers.BatchNormalization()(l12)
le1 = tf.keras.layers.Dense(512, activation="gelu")(bn1)
bn2 = tf.keras.layers.BatchNormalization()(le1)
le2 = tf.keras.layers.Dense(512, activation="gelu")(bn2)
bn3 = tf.keras.layers.BatchNormalization()(le2)
le3 = tf.keras.layers.Dense(512, activation="gelu")(bn3)
bn4 = tf.keras.layers.BatchNormalization()(le3)
le4 = tf.keras.layers.Dense(256, activation="gelu")(bn4)
bn_5 = tf.keras.layers.BatchNormalization()(le4)
le5 = tf.keras.layers.Dense(256, activation="gelu")(bn_5)
bn_6 = tf.keras.layers.BatchNormalization()(le5)

le6 = tf.keras.layers.Dense(128, activation="gelu")(bn_6)
bn5 = tf.keras.layers.BatchNormalization()(le6)
le9 = tf.keras.layers.Dense(64, activation="gelu")(bn5)
bn6 = tf.keras.layers.BatchNormalization()(le9)
l13 = tf.keras.layers.Dense(16, activation="gelu")(bn6)
bn7 = tf.keras.layers.BatchNormalization()(l13)
l14 = tf.keras.layers.Dense(1, activation='sigmoid')(bn7)
output = l14*tf.constant([100],dtype=tf.float32)

model = tf.keras.Model(inputs={"Image":input_image,"Feature":input_meta},outputs={"Label":output})


ValueError: Cannot convert '12' to a shape.

In [7]:
model.summary()


NameError: name 'model' is not defined

In [8]:
initial_learning_rate = 2.0
first_decay_steps=100
t_mul=2.0
m_mul=0.5
alpha=10**(-3)
lr_warmup_decayed_fn = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate, first_decay_steps, 
    t_mul=t_mul,
    m_mul=m_mul,
    alpha=alpha
)


In [9]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=10**(-3)), loss=tf.keras.losses.MeanSquaredError(), metrics=[tf.keras.metrics.RootMeanSquaredError()])


NameError: name 'model' is not defined

In [10]:
gen_train = train_loader.batch(32).repeat()
gen_train = gen_train.prefetch(100)

gen_test = test_loader.batch(32)
gen_test = gen_test.prefetch(100)


In [11]:
model.fit(gen_train, steps_per_epoch=len(os.listdir("/kaggle/input/petfinder-pawpularity-score/train/"))*0.95//32 , epochs=2, validation_data=gen_test, validation_steps=1)


NameError: name 'model' is not defined

In [12]:
def pythonic_loader_test():
    IMAGES_PATH = "/kaggle/input/petfinder-pawpularity-score/test/"
    CSV_PATH = "/kaggle/input/petfinder-pawpularity-score/test.csv"
    
    relevant_columns = ['Subject Focus','Eyes','Face','Near','Action','Accessory','Group','Collage','Human','Occlusion','Info','Blur']
    
    image_names = os.listdir(IMAGES_PATH)
    ###image_names = image_names[:int(len(image_names)*0.95)]
    ###np.random.shuffle(image_names)
    metadata_csv = pd.read_csv(CSV_PATH)
    
    for i in image_names:
        img = tf.keras.utils.load_img(path = IMAGES_PATH+i,
                color_mode="rgb",
                target_size=(256,256))
        img = np.array(img)
        
        metadata = metadata_csv[metadata_csv['Id']==i[:-4]]
        features = metadata[relevant_columns].values[0]
        
        yield({"Image":img,"Feature":features})


In [13]:
test_loader = tf.data.Dataset.from_generator(
            pythonic_loader_test,
            output_types=({'Image': tf.int64,
            'Feature': tf.int64}))


In [14]:
preds = model.predict(test_loader.batch(1))


NameError: name 'model' is not defined

In [15]:
ser = pd.Series(preds["Label"].squeeze(),name="Pawpularity")


NameError: name 'preds' is not defined

In [16]:
id_ = pd.Series([i[:-4] for i in os.listdir("/kaggle/input/petfinder-pawpularity-score/test/")],name="Id")


In [17]:
ans = pd.DataFrame(id_).join(ser)


NameError: name 'ser' is not defined

In [18]:
ans.to_csv("submission.csv", index=False)


NameError: name 'ans' is not defined